# RL Traffic Light Control – Bougara El Biar Intersection (SUMO + PPO)
This notebook installs SUMO and all RL dependencies, runs the simulation, and trains an autonomous traffic light controller using **RecurrentPPO (PPO + LSTM)**.

Optimized for **Kaggle Notebooks** (as well as local Windows/Linux machines).

## 1. Install Dependencies & SUMO

In [ ]:
import os, sys
ON_KAGGLE = os.path.exists("/kaggle")

if ON_KAGGLE:
    print("[INFO] Running on Kaggle! Installing SUMO system binaries...")
    !add-apt-repository -y ppa:sumo/stable
    !apt-get update -y
    !apt-get install -y sumo sumo-tools sumo-doc
    os.environ["SUMO_HOME"] = "/usr/share/sumo"
    os.environ["PATH"] += ":/usr/share/sumo/bin"
else:
    print("[INFO] Running locally.")

# Install Python RL dependencies
!pip install --quiet "stable-baselines3[extra]>=2.0" gymnasium traci "sb3-contrib>=2.0" matplotlib

# Verify SUMO is recognized
!sumo --version

## 2. Sync Files (Kaggle Dataset -> /kaggle/working)

In [ ]:
import os, shutil, glob

if ON_KAGGLE:
    working_dir = "/kaggle/working"
    sumo_dir = os.path.join(working_dir, "sumo_files")
    os.makedirs(sumo_dir, exist_ok=True)

    # Locate uploaded dataset in /kaggle/input
    candidates = [
        "/kaggle/input/datasets/rayantribeche/bougara-rl",
        "/kaggle/input/bougara-rl",
    ] + glob.glob("/kaggle/input/**/bougara.sumocfg", recursive=True)

    dataset_dir = None
    for c in candidates:
        d = os.path.dirname(c) if c.endswith(".sumocfg") else c
        if os.path.exists(d) and os.path.exists(os.path.join(d, "bougara.sumocfg")):
            dataset_dir = d
            break

    if dataset_dir:
        print(f"[INFO] Found dataset at: {dataset_dir}")
        for fname in os.listdir(dataset_dir):
            src = os.path.join(dataset_dir, fname)
            if fname.endswith((".xml", ".sumocfg")):
                shutil.copy(src, os.path.join(sumo_dir, fname))
            elif fname.endswith(".py"):
                shutil.copy(src, os.path.join(working_dir, fname))
        print("[SUCCESS] Project files synced to /kaggle/working!")
    else:
        print("[INFO] No external dataset detected. Using files currently in /kaggle/working.")

print("Working directory contents:", os.listdir("."))

## 3. Test Traffic Generator
Verifies that vehicle routes, realistic heterogeneous speed distributions, and traffic scenarios generate cleanly.

In [ ]:
import traffic_generator

sumo_dir = "sumo_files"
traffic_generator.generate_traffic(sumo_dir, scenario="random")
print("[OK] Traffic routes generated successfully!")

## 4. Train the Reinforcement Learning Model
* **Algorithm**: RecurrentPPO (PPO with LSTM memory)
* **Observations**: 29 features (queues, waiting times, vehicle speed ratios, green phases)
* **Step Architecture**: Macro-decisions (5s green extension or 13s transition + guaranteed min green)
* **Early Stopping**: Patience of 10 evaluations (250,000 steps of stagnation) to stop when fully converged and save GPU hours.

In [ ]:
import rl

# Start training (will run up to 500,000 steps or auto-stop upon convergence)
model = rl.train(
    total_timesteps = 500_000,
    n_eval_episodes = 5,
    save_dir        = "models",
    log_dir         = "logs",
)

## 5. View Training Reward Curves

In [ ]:
from IPython.display import Image, display

curve_path = "logs/reward_curve.png"
if os.path.exists(curve_path):
    display(Image(filename=curve_path))
else:
    print("Reward curve will appear here once training completes.")

## 6. Evaluate the Best Saved Model

In [ ]:
# Evaluate best model
best_model_path = "models/best/best_model.zip"
if os.path.exists(best_model_path):
    print(f"Evaluating: {best_model_path}")
    rl.evaluate(model_path=best_model_path, n_episodes=5, use_gui=False)
else:
    final_model_path = "models/bougara_lstm_final.zip"
    if os.path.exists(final_model_path):
        rl.evaluate(model_path=final_model_path, n_episodes=5, use_gui=False)

## 7. Package Results for 1-Click Kaggle Download
Creates `bougara_results.zip` containing all checkpoints, normalization files (`vec_normalize.pkl`), and logs.

In [ ]:
!zip -q -r bougara_results.zip models/ logs/ sumo_files/
print("[SUCCESS] Created bougara_results.zip!")
print("Look at the right sidebar under 'Output' to download the zip file.")